# AlphaFold-Multimer: Protein Complex Structure Prediction

**Goal**: Predict the 3D structure of a heteromeric protein complex for downstream molecular docking.

**Input**:
- Chain A FASTA (e.g., host protein)
- Chain B FASTA (e.g., pathogen protein)

**Output**: Predicted complex PDB + PAE matrix → for AutoDock Vina Grid Box


## 1. Prepare Multi-Chain FASTA

Merge Chain A and Chain B sequences into one FASTA entry, separated by `:`.

In [ ]:
# === USER: Replace these with your actual sequences ===
chain_a_fasta = ">ChainA\nYOUR_SEQUENCE_HERE"
chain_b_fasta = ">ChainB\nYOUR_SEQUENCE_HERE"

# Merge for AlphaFold-Multimer
combined_fasta = chain_a_fasta + chain_b_fasta

with open('complex.fasta', 'w') as f:
    f.write(combined_fasta)

print('complex.fasta written.')
print(combined_fasta)

## 2. Run AlphaFold-Multimer (ColabFold)

Install ColabFold and run prediction.

In [ ]:
# Install ColabFold (run once per Colab session)
!pip install -q colabfold

import os
os.makedirs('AF_output', exist_ok=True)

from colabfold.batch import run
run(
    queries=['complex.fasta'],
    result_dir='./AF_output',
    model_type='alphafold2_multimer_v3',
    num_models=5,
    num_recycles=6,
    is_complex=True,
)

## 3. Model Quality: pLDDT + PAE

**pLDDT** > 70: Confident fold

**PAE** (interface) < 10 Å: Proceed with docking
**PAE** > 15 Å: Interface unreliable

In [ ]:
import glob
import numpy as np
import json

# Find best model
pdb_paths = sorted(glob.glob('AF_output/*_rank_001*.pdb'))
pae_paths = sorted(glob.glob('AF_output/*_aligned_error*.json'))

best_pdb = pdb_paths[0] if pdb_paths else None
best_pae = pae_paths[0] if pae_paths else None

print(f'Best model: {best_pdb}')
print(f'PAE file:   {best_pae}')

In [ ]:
# Extract pLDDT per chain from PDB
def parse_plddt(pdb_path):
    plddt_by_chain = {}
    for line in open(pdb_path):
        if not line.startswith('ATOM'): continue
        if line[12:16].strip() != 'CA': continue
        chain = line[21]
        plddt = float(line[60:66])
        plddt_by_chain.setdefault(chain, []).append(plddt)
    return plddt_by_chain

plddt = parse_plddt(best_pdb)
for ch, vals in plddt.items():
    print(f'Chain {ch}: pLDDT mean={np.mean(vals):.1f}, >70: {sum(1 for v in vals if v>70)}/{len(vals)}')

In [ ]:
# Analyze PAE matrix
with open(best_pae) as f:
    pae_raw = json.load(f)

pae = np.array(pae_raw['predicted_aligned_error'])
n = pae.shape[0]
mid = n // 2

pae_inter = (pae[:mid, mid:].mean() + pae[mid:, :mid].mean()) / 2

print(f'Interface PAE: {pae_inter:.2f} A')
if pae_inter < 10:
    print('RELIABLE: Proceed with docking')
elif pae_inter < 15:
    print('MODERATE: Interpret with caution')
else:
    print('LOW confidence: Docking may not be meaningful')

## 4. Calculate Vina Grid Box

Grid box center = centroid of interface CA atoms.

In [ ]:
# Calculate interface centroid from PDB
ca_a = {}
ca_b = {}
for line in open(best_pdb):
    if not line.startswith('ATOM'): continue
    if line[12:16].strip() != 'CA': continue
    ch = line[21]
    res = int(line[22:26])
    coords = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    if ch == 'A': ca_a[res] = coords
    elif ch == 'B': ca_b[res] = coords

# Find interface (CA < 8 A between chains)
interface_ca = []
for r1, p1 in ca_a.items():
    for r2, p2 in ca_b.items():
        if np.linalg.norm(p1 - p2) < 8.0:
            interface_ca.append(p1)
            interface_ca.append(p2)
            break

center = np.mean(interface_ca, axis=0)
grid_size = 22

print(f'Vina Grid Box:')
print(f'  center_x = {center[0]:.1f}')
print(f'  center_y = {center[1]:.1f}')
print(f'  center_z = {center[2]:.1f}')
print(f'  size_x = size_y = size_z = {grid_size}')

# Save config
with open('docking_config.txt', 'w') as f:
    f.write(f'center_x = {center[0]:.1f}\n')
    f.write(f'center_y = {center[1]:.1f}\n')
    f.write(f'center_z = {center[2]:.1f}\n')
    f.write(f'size = {grid_size}\n')

print('\ndocking_config.txt saved.')

## 5. Download Results

Download from Colab:
- `AF_output/*_rank_001*.pdb` → receptor for Vina
- `AF_output/*_aligned_error*.json` → PAE data
- `docking_config.txt` → Grid Box parameters

Then run AutoDock Vina on your local machine (WSL/Linux) with these inputs.